# Member 04: Amazon Athena & Business Analytics
### Project: AWS-Based Customer Churn Prediction System
**Objective**: Query model predictions and customer features using serverless SQL (Amazon Athena) to uncover churn drivers, assess revenue at risk, and generate actionable business retention strategies.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Scored Holdout Predictions
We load the predictions generated by Member 03's champion `HistGradientBoostingClassifier`.

In [ ]:
data_path = '../datasets/predictions/cell2cellholdout_predictions.csv'
if not os.path.exists(data_path):
    data_path = 'datasets/predictions/cell2cellholdout_predictions.csv'

df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} customer records across {len(df.columns)} features.")
df[['CustomerID', 'ServiceArea', 'MonthlyRevenue', 'ChurnPrediction', 'ChurnProbability', 'RiskSegment']].head()

## 2. In-Memory SQL Engine (Athena Simulation)
We mirror Amazon Athena's serverless SQL execution using an in-memory SQL database.

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('customer_predictions', conn, index=False, if_exists='replace')
print("SQL Table customer_predictions registered successfully!")

## 3. Query 1: Overall Risk Segmentation & Revenue at Risk

In [ ]:
q1_sql = """
SELECT 
    RiskSegment,
    COUNT(*) AS CustomerCount,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customer_predictions), 2) AS PercentageOfTotal,
    ROUND(AVG(ChurnProbability) * 100, 2) AS AvgChurnProbabilityPct,
    ROUND(SUM(MonthlyRevenue), 2) AS TotalMonthlyRevenueAtRisk
FROM customer_predictions
GROUP BY RiskSegment
ORDER BY AvgChurnProbabilityPct DESC;
"""
q1_df = pd.read_sql_query(q1_sql, conn)
display(q1_df)

In [ ]:
# Visualizing Risk Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = {'High Risk': '#d9534f', 'Medium Risk': '#f0ad4e', 'Low Risk': '#5cb85c'}
bar_colors = [colors[s] for s in q1_df['RiskSegment']]

ax1.pie(q1_df['CustomerCount'], labels=q1_df['RiskSegment'], autopct='%1.1f%%', colors=bar_colors, explode=(0.08, 0, 0))
ax1.set_title('Customer Distribution by Risk Tier', fontweight='bold')

bars = ax2.bar(q1_df['RiskSegment'], q1_df['TotalMonthlyRevenueAtRisk'] / 1000, color=bar_colors)
ax2.set_title('Monthly Revenue At Risk ($ in Thousands)', fontweight='bold')
ax2.set_ylabel('$ Thousands')
for b in bars:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 5, f'${b.get_height():,.1f}k', ha='center')
plt.show()

## 4. Query 2: Behavioral & Usage Profile Across Risk Tiers

In [ ]:
q2_sql = """
SELECT 
    RiskSegment,
    COUNT(*) AS CustomerCount,
    ROUND(AVG(MonthlyRevenue), 2) AS AvgMonthlyRevenue,
    ROUND(AVG(MonthlyMinutes), 1) AS AvgMonthlyMinutes,
    ROUND(AVG(TotalRecurringCharge), 2) AS AvgRecurringCharge,
    ROUND(AVG(OverageMinutes), 1) AS AvgOverageMinutes,
    ROUND(AVG(CurrentEquipmentDays), 0) AS AvgEquipmentAgeDays
FROM customer_predictions
GROUP BY RiskSegment
ORDER BY AvgMonthlyRevenue DESC;
"""
display(pd.read_sql_query(q2_sql, conn))

## 5. Query 3: Geographical Churn Concentration by Service Area

In [ ]:
q3_sql = """
SELECT 
    ServiceArea,
    COUNT(*) AS TotalCustomers,
    SUM(CASE WHEN ChurnPrediction = 1 THEN 1 ELSE 0 END) AS PredictedChurners,
    ROUND(AVG(ChurnPrediction) * 100, 2) AS ChurnRatePct,
    SUM(CASE WHEN RiskSegment = 'High Risk' THEN 1 ELSE 0 END) AS HighRiskCustomers,
    ROUND(AVG(ChurnProbability) * 100, 2) AS AvgRiskProbabilityPct
FROM customer_predictions
WHERE ServiceArea IS NOT NULL AND ServiceArea != 'UNKNOWN'
GROUP BY ServiceArea
HAVING COUNT(*) >= 50
ORDER BY HighRiskCustomers DESC
LIMIT 10;
"""
q3_df = pd.read_sql_query(q3_sql, conn)
display(q3_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=q3_df, x='HighRiskCustomers', y='ServiceArea', palette='Blues_r')
plt.title('Top 10 Telecom Service Areas by High-Risk Customer Count', fontweight='bold')
plt.show()

## 6. Query 4: Impact of Device Age and Handset Condition

In [ ]:
q4_sql = """
SELECT 
    CASE 
        WHEN CurrentEquipmentDays < 180 THEN '1. Under 6 Months'
        WHEN CurrentEquipmentDays BETWEEN 180 AND 365 THEN '2. 6 - 12 Months'
        WHEN CurrentEquipmentDays BETWEEN 366 AND 730 THEN '3. 1 - 2 Years'
        ELSE '4. Over 2 Years'
    END AS EquipmentAgeBracket,
    HandsetRefurbished,
    COUNT(*) AS CustomerCount,
    ROUND(AVG(ChurnProbability) * 100, 2) AS AvgChurnRiskPct
FROM customer_predictions
GROUP BY 1, 2
ORDER BY 1, 2;
"""
q4_df = pd.read_sql_query(q4_sql, conn)
display(q4_df)

## 7. Retention Recommendations Summary
1. **365-Day Handset Upgrade Campaign**: Offer device upgrades to customers before phone age exceeds 1 year.
2. **VIP Concierge**: Outreach to the top 50 high-revenue accounts in High-Risk status.
3. **Regional Marketing**: Launch promotional bundling in New York, Dallas, and Los Angeles metro areas.